# Miniprot vs TBLASTN Window Test for 001.Allagelena_difficilis

This notebook tests how full-length spidroin proteins align to candidate genomic windows with two approaches:

1. **miniprot** protein-to-genome spliced alignment, emitted as mixed GFF/PAF output and then split into separate files.
2. **TBLASTN** protein-vs-nucleotide local HSP search, used as a contrast rather than as a gene model.

The candidate windows come from existing genome-coordinate nHMMER and miniprot hits. Each hit is expanded by 50 kb on both sides, clipped to contig boundaries, merged with `bedtools`, and extracted with `seqkit`.

## Environment Setup

Run the notebook from the project root with the Pixi environment available:

```bash
pixi install
```

Expected command-line tools: `bedtools`, `seqkit`, `miniprot`, `makeblastdb`, and `tblastn`.


## Configuration

Import packages and predefine input/output paths used throughout the notebook.


In [ ]:
from pathlib import Path
import shlex

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import polars as pl

from spider_silkome_module import (
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
)
from spider_silkome_module.utils.run_cmd import run_cmd


In [ ]:
TASK_NAME = "miniprot_blast_window_test_20260523"
SPECIES = "001.Allagelena_difficilis"
THREADS = 16
FORCE = False
FLANK_BP = 50000

species_raw_dir = RAW_DATA_DIR / "01.ref_gff" / SPECIES
GENOME_FASTA = species_raw_dir / "LAg-10-m5.genome.fa.gz"
GENOME_FAI = Path(str(GENOME_FASTA) + ".fai")

NHMMER_GFF = PROCESSED_DATA_DIR / "nhmmer_search_parsed" / SPECIES / f"{SPECIES}.gff"
MINIPROT_EVIDENCE_GFF = PROCESSED_DATA_DIR / "miniprot_output" / f"{SPECIES}.gff"
QUERY_PROTEINS = EXTERNAL_DATA_DIR / "spidroins_full_length_with_dataset1_rename.fasta"

OUT_DIR = INTERIM_DATA_DIR / TASK_NAME / SPECIES
BLAST_DB_DIR = OUT_DIR / "blastdb"

raw_bed = OUT_DIR / "candidate_windows.raw.bed"
merged_bed = OUT_DIR / "candidate_windows.merged.bed"
window_fasta = OUT_DIR / "candidate_windows.fa"

miniprot_mixed = OUT_DIR / "miniprot_vs_windows.gff"
miniprot_gff = OUT_DIR / "miniprot_vs_windows.only.gff3"
miniprot_paf = OUT_DIR / "miniprot_vs_windows.paf"
miniprot_unclassified = OUT_DIR / "miniprot_vs_windows.unclassified.txt"

tblastn_tsv = OUT_DIR / "tblastn_vs_windows.tsv"
blast_db_prefix = BLAST_DB_DIR / "windows"

for p in [GENOME_FASTA, GENOME_FAI, NHMMER_GFF, MINIPROT_EVIDENCE_GFF, QUERY_PROTEINS]:
    if not p.exists():
        raise FileNotFoundError(p)

OUT_DIR.mkdir(parents=True, exist_ok=True)
BLAST_DB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT_DIR}")


## Custom Functions

Notebook-specific parsers and small helpers. Reusable production logic should be moved into `spider_silkome_module` later if this experiment becomes part of the pipeline.


In [ ]:
def parse_attrs(attr_str: str) -> dict[str, str]:
    attrs: dict[str, str] = {}
    for part in attr_str.strip().split(";"):
        if "=" in part:
            key, value = part.split("=", 1)
            attrs[key.strip()] = value.strip()
    return attrs


def read_fai(fai_path: Path) -> dict[str, int]:
    lengths: dict[str, int] = {}
    with fai_path.open() as fh:
        for line in fh:
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 2:
                lengths[fields[0]] = int(fields[1])
    return lengths


def parse_evidence_intervals(
    gff_path: Path,
    source_label: str,
    features: set[str],
    contig_lengths: dict[str, int],
    flank_bp: int,
) -> list[tuple[str, int, int, str]]:
    """Return BED intervals as (chrom, start0, end0, label)."""
    intervals: list[tuple[str, int, int, str]] = []
    with gff_path.open() as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue
            chrom, _src, feature, start_s, end_s, score, strand, _phase, attrs_s = fields[:9]
            if feature not in features or chrom not in contig_lengths:
                continue
            start = int(start_s)
            end = int(end_s)
            bed_start = max(0, start - 1 - flank_bp)
            bed_end = min(contig_lengths[chrom], end + flank_bp)
            attrs = parse_attrs(attrs_s)
            name = attrs.get("Name") or attrs.get("ID") or attrs.get("Target", ".").split()[0] or feature
            label = f"{source_label}:{feature}:{name}:{strand}:{score}"
            if bed_end > bed_start:
                intervals.append((chrom, bed_start, bed_end, label))
    return intervals


def write_bed(path: Path, intervals: list[tuple[str, int, int, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as out:
        for chrom, start, end, label in intervals:
            out.write(f"{chrom}\t{start}\t{end}\t{label}\n")


def shell_path(path: Path) -> str:
    return shlex.quote(str(path))


def is_int_string(value: str) -> bool:
    try:
        int(value)
        return True
    except ValueError:
        return False


GFF_FEATURES = {
    "gene", "mRNA", "CDS", "exon", "intron", "start_codon", "stop_codon",
    "five_prime_UTR", "three_prime_UTR",
}


def is_gff_line(fields: list[str]) -> bool:
    return len(fields) >= 9 and fields[2] in GFF_FEATURES


def is_paf_line(fields: list[str]) -> bool:
    if len(fields) < 12 or fields[4] not in {"+", "-"}:
        return False
    int_cols = [1, 2, 3, 6, 7, 8, 9, 10, 11]
    return all(is_int_string(fields[i]) for i in int_cols)


def split_miniprot_mixed_output(
    mixed_path: Path,
    gff_path: Path,
    paf_path: Path,
    unclassified_path: Path,
) -> dict[str, int]:
    counts = {"gff": 0, "paf": 0, "unclassified": 0}
    with mixed_path.open() as inp, gff_path.open("w") as gff, paf_path.open("w") as paf, unclassified_path.open("w") as unk:
        for line in inp:
            if not line.strip():
                continue
            if line.startswith("#"):
                gff.write(line)
                counts["gff"] += 1
                continue
            fields = line.rstrip("\n").split("\t")
            if is_gff_line(fields):
                gff.write(line)
                counts["gff"] += 1
            elif is_paf_line(fields):
                paf.write(line)
                counts["paf"] += 1
            else:
                unk.write(line)
                counts["unclassified"] += 1
    return counts


def load_paf(path: Path) -> pl.DataFrame:
    columns = [
        "query", "qlen", "qstart", "qend", "strand", "target", "tlen", "tstart", "tend",
        "nmatch", "alen", "mapq",
    ]
    rows = []
    if not path.exists() or path.stat().st_size == 0:
        return pl.DataFrame(schema={c: pl.Utf8 for c in columns})
    with path.open() as fh:
        for line in fh:
            fields = line.rstrip("\n").split("\t")
            if not is_paf_line(fields):
                continue
            row = dict(zip(columns, fields[:12]))
            row["tags"] = ";".join(fields[12:])
            rows.append(row)
    if not rows:
        return pl.DataFrame(schema={c: pl.Utf8 for c in columns + ["tags"]})
    return pl.DataFrame(rows).with_columns([
        pl.col(c).cast(pl.Int64) for c in ["qlen", "qstart", "qend", "tlen", "tstart", "tend", "nmatch", "alen", "mapq"]
    ]).with_columns([
        ((pl.col("qend") - pl.col("qstart")) / pl.col("qlen")).alias("query_coverage"),
        (pl.col("tend") - pl.col("tstart")).abs().alias("target_span"),
    ])


def load_tblastn(path: Path) -> pl.DataFrame:
    columns = [
        "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend",
        "sstart", "send", "evalue", "bitscore", "qlen", "slen", "sframe",
    ]
    if not path.exists() or path.stat().st_size == 0:
        return pl.DataFrame(schema={c: pl.Utf8 for c in columns})
    return pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        new_columns=columns,
        schema_overrides={
            "pident": pl.Float64,
            "length": pl.Int64,
            "mismatch": pl.Int64,
            "gapopen": pl.Int64,
            "qstart": pl.Int64,
            "qend": pl.Int64,
            "sstart": pl.Int64,
            "send": pl.Int64,
            "evalue": pl.Float64,
            "bitscore": pl.Float64,
            "qlen": pl.Int64,
            "slen": pl.Int64,
            "sframe": pl.Int64,
        },
    ).with_columns([
        ((pl.col("qend") - pl.col("qstart") + 1).abs() / pl.col("qlen")).alias("query_coverage"),
        (pl.col("send") - pl.col("sstart") + 1).abs().alias("subject_span"),
    ])


def summarize_miniprot_gff(path: Path) -> tuple[pl.DataFrame, pl.DataFrame]:
    mrna_rows = []
    cds_counts: dict[str, int] = {}
    if not path.exists() or path.stat().st_size == 0:
        return pl.DataFrame(), pl.DataFrame()
    with path.open() as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue
            seqid, source, feature, start_s, end_s, score, strand, phase, attrs_s = fields[:9]
            attrs = parse_attrs(attrs_s)
            if feature == "mRNA":
                mrna_id = attrs.get("ID", "")
                target = attrs.get("Target", "").split()
                mrna_rows.append({
                    "mrna_id": mrna_id,
                    "seqid": seqid,
                    "start": int(start_s),
                    "end": int(end_s),
                    "score": float(score) if score != "." else None,
                    "strand": strand,
                    "query": target[0] if target else "",
                    "query_from": int(target[1]) if len(target) > 1 and target[1].isdigit() else None,
                    "query_to": int(target[2]) if len(target) > 2 and target[2].isdigit() else None,
                    "identity": float(attrs.get("Identity", "nan")) if attrs.get("Identity") else None,
                    "rank": int(attrs.get("Rank", "0")) if attrs.get("Rank", "0").isdigit() else None,
                })
            elif feature == "CDS":
                parent = attrs.get("Parent", "")
                cds_counts[parent] = cds_counts.get(parent, 0) + 1
    mrna_df = pl.DataFrame(mrna_rows) if mrna_rows else pl.DataFrame()
    cds_df = pl.DataFrame([{"mrna_id": k, "cds_blocks": v} for k, v in cds_counts.items()]) if cds_counts else pl.DataFrame()
    if not mrna_df.is_empty() and not cds_df.is_empty():
        mrna_df = mrna_df.join(cds_df, on="mrna_id", how="left").with_columns(pl.col("cds_blocks").fill_null(0))
    return mrna_df, cds_df


## Step 1: Build Candidate Windows

Collect genome-coordinate evidence from nHMMER (`NTD`/`CTD`) and the existing genome-level miniprot GFF (`mRNA`), expand each interval by ±50 kb, clip to contig boundaries, and merge overlapping windows.


In [ ]:
contig_lengths = read_fai(GENOME_FAI)
nhmmer_intervals = parse_evidence_intervals(
    NHMMER_GFF,
    source_label="nhmmer",
    features={"NTD", "CTD"},
    contig_lengths=contig_lengths,
    flank_bp=FLANK_BP,
)
miniprot_intervals = parse_evidence_intervals(
    MINIPROT_EVIDENCE_GFF,
    source_label="miniprot",
    features={"mRNA"},
    contig_lengths=contig_lengths,
    flank_bp=FLANK_BP,
)
all_intervals = nhmmer_intervals + miniprot_intervals
write_bed(raw_bed, all_intervals)

print(f"nHMMER intervals: {len(nhmmer_intervals)}")
print(f"miniprot mRNA intervals: {len(miniprot_intervals)}")
print(f"raw window intervals: {len(all_intervals)} -> {raw_bed}")


In [ ]:
merge_cmd = (
    f"pixi run bedtools sort -i {shell_path(raw_bed)} | "
    f"pixi run bedtools merge -i - -c 4 -o distinct,count > {shell_path(merged_bed)}"
)
run_cmd(merge_cmd, [merged_bed], force=FORCE)

merged = pl.read_csv(
    merged_bed,
    separator="\t",
    has_header=False,
    new_columns=["chrom", "start0", "end0", "sources", "n_raw_intervals"],
    schema_overrides={"start0": pl.Int64, "end0": pl.Int64, "n_raw_intervals": pl.Int64},
)
merged = merged.with_columns((pl.col("end0") - pl.col("start0")).alias("window_len"))
print(f"merged windows: {merged.height}")
print(f"total merged bp: {merged.get_column('window_len').sum():,}")
merged.head(10)


## Step 2: Extract Window FASTA

Use `seqkit subseq --bed` to extract the merged candidate windows from the genome FASTA.


In [ ]:
extract_cmd = (
    f"pixi run seqkit subseq --bed {shell_path(merged_bed)} "
    f"{shell_path(GENOME_FASTA)} -o {shell_path(window_fasta)}"
)
run_cmd(extract_cmd, [window_fasta], force=FORCE)

window_size = window_fasta.stat().st_size if window_fasta.exists() else 0
print(f"window FASTA: {window_fasta}")
print(f"size: {window_size:,} bytes")
if window_size == 0:
    raise RuntimeError("candidate window FASTA is empty")


## Step 3: Run Miniprot

Run miniprot on the extracted genomic windows and emit mixed GFF/PAF output. Do not use `-S`; that option disables spliced alignment.


In [ ]:
miniprot_cmd = (
    f"pixi run miniprot --gff -t {THREADS} -G 200000 --outc 0.05 --outs 0.10 --outn 1000 "
    f"{shell_path(window_fasta)} {shell_path(QUERY_PROTEINS)} > {shell_path(miniprot_mixed)}"
)
run_cmd(miniprot_cmd, [miniprot_mixed], force=FORCE)

print(f"miniprot mixed output: {miniprot_mixed}")
print(f"size: {miniprot_mixed.stat().st_size:,} bytes")


## Step 4: Split Miniprot Mixed GFF/PAF Output

Split the mixed miniprot output into a clean GFF3-like file, a PAF file, and an unclassified file for any unexpected non-empty lines.


In [ ]:
if FORCE or not (miniprot_gff.exists() and miniprot_paf.exists() and miniprot_unclassified.exists()):
    split_counts = split_miniprot_mixed_output(
        miniprot_mixed,
        miniprot_gff,
        miniprot_paf,
        miniprot_unclassified,
    )
else:
    split_counts = {
        "gff": sum(1 for _ in miniprot_gff.open()),
        "paf": sum(1 for _ in miniprot_paf.open()),
        "unclassified": sum(1 for _ in miniprot_unclassified.open()),
    }

print(split_counts)
print(f"GFF3-only: {miniprot_gff}")
print(f"PAF-only:  {miniprot_paf}")
print(f"unclassified: {miniprot_unclassified}")

if miniprot_paf.stat().st_size == 0:
    print("Note: PAF split is empty. This miniprot version/parameter set may emit only GFF lines with --gff.")


## Step 5: Run TBLASTN

Build a nucleotide BLAST database from the same extracted windows and run full-length spidroin proteins against it with `tblastn`.


In [ ]:
makeblastdb_cmd = (
    f"pixi run makeblastdb -in {shell_path(window_fasta)} -dbtype nucl "
    f"-parse_seqids -out {shell_path(blast_db_prefix)}"
)
run_cmd(makeblastdb_cmd, [Path(str(blast_db_prefix) + ".nin")], force=FORCE)

tblastn_cmd = (
    f"pixi run tblastn -query {shell_path(QUERY_PROTEINS)} -db {shell_path(blast_db_prefix)} "
    f"-evalue 1e-5 -seg no -num_threads {THREADS} "
    "-outfmt '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qlen slen sframe' "
    f"-out {shell_path(tblastn_tsv)}"
)
run_cmd(tblastn_cmd, [tblastn_tsv], force=FORCE)

print(f"TBLASTN output: {tblastn_tsv}")
print(f"size: {tblastn_tsv.stat().st_size:,} bytes")


## Results

Load the miniprot GFF/PAF split files and the TBLASTN HSP table, then summarize how many alignments each tool reports. TBLASTN rows are local HSPs, not exon/intron gene models.


In [ ]:
miniprot_mrna_df, miniprot_cds_df = summarize_miniprot_gff(miniprot_gff)
miniprot_paf_df = load_paf(miniprot_paf)
tblastn_df = load_tblastn(tblastn_tsv)

print("Window summary")
print(f"  merged windows: {merged.height}")
print(f"  total merged bp: {merged.get_column('window_len').sum():,}")
print("Miniprot summary")
print(f"  mRNA rows: {miniprot_mrna_df.height if not miniprot_mrna_df.is_empty() else 0}")
print(f"  CDS parent rows: {miniprot_cds_df.height if not miniprot_cds_df.is_empty() else 0}")
print(f"  PAF rows: {miniprot_paf_df.height if not miniprot_paf_df.is_empty() else 0}")
print("TBLASTN summary")
print(f"  HSP rows: {tblastn_df.height if not tblastn_df.is_empty() else 0}")

if not miniprot_mrna_df.is_empty():
    display(miniprot_mrna_df.sort("score", descending=True).head(10))
if not miniprot_paf_df.is_empty():
    display(miniprot_paf_df.sort("query_coverage", descending=True).head(10))
if not tblastn_df.is_empty():
    display(tblastn_df.sort("bitscore", descending=True).head(10))


## Visual Comparison

Draw a compact track for one selected extracted window. Miniprot GFF mRNA/CDS rows are shown as gene-model-like features; TBLASTN rows are shown as local HSP blocks.


In [ ]:
def plot_window_comparison(window_id: str | None = None, top_hsps: int = 100):
    if miniprot_mrna_df.is_empty() and tblastn_df.is_empty():
        raise ValueError("No miniprot or TBLASTN rows to plot")
    if window_id is None:
        if not miniprot_mrna_df.is_empty():
            window_id = miniprot_mrna_df.sort("score", descending=True).item(0, "seqid")
        else:
            window_id = tblastn_df.sort("bitscore", descending=True).item(0, "sseqid")

    fig, ax = plt.subplots(figsize=(14, 4))
    y_miniprot = 1.0
    y_tblastn = 0.0

    if not miniprot_mrna_df.is_empty():
        mp = miniprot_mrna_df.filter(pl.col("seqid") == window_id).sort("score", descending=True).head(20)
        for row in mp.iter_rows(named=True):
            x0 = min(row["start"], row["end"])
            width = abs(row["end"] - row["start"]) + 1
            ax.add_patch(Rectangle((x0, y_miniprot - 0.18), width, 0.28, color="#2563eb", alpha=0.45))

    if not tblastn_df.is_empty():
        tb = tblastn_df.filter(pl.col("sseqid") == window_id).sort("bitscore", descending=True).head(top_hsps)
        for row in tb.iter_rows(named=True):
            x0 = min(row["sstart"], row["send"])
            width = abs(row["send"] - row["sstart"]) + 1
            ax.add_patch(Rectangle((x0, y_tblastn - 0.12), width, 0.20, color="#dc2626", alpha=0.35))

    ax.set_yticks([y_tblastn, y_miniprot])
    ax.set_yticklabels(["TBLASTN HSP", "miniprot mRNA"])
    ax.set_xlabel("Position on extracted window sequence (bp)")
    ax.set_title(window_id)
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    return fig, ax

# Example:
fig, ax = plot_window_comparison()


In [ ]:
from pathlib import Path
from collections import defaultdict
import polars as pl

gff = miniprot_mixed
query_fasta = QUERY_PROTEINS

def fasta_lengths(path):
    lengths = {}
    name = None
    n = 0
    for line in path.read_text().splitlines():
        if line.startswith(">"):
            if name is not None:
                lengths[name] = n
            name = line[1:].split()[0]
            n = 0
        else:
            n += len(line.strip())
    if name is not None:
        lengths[name] = n
    return lengths

def parse_attrs(s):
    out = {}
    for part in s.split(";"):
        if "=" in part:
            k, v = part.split("=", 1)
            out[k] = v
    return out

query_len = fasta_lengths(query_fasta)
cds_count = defaultdict(int)
mrnas = []

with gff.open() as fh:
    for line in fh:
        if not line.strip() or line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue

        seqid, source, feature, start, end, score, strand, phase, attr = fields
        attrs = parse_attrs(attr)

        if feature == "CDS":
            cds_count[attrs.get("Parent")] += 1

        if feature == "mRNA":
            target = attrs.get("Target", "").split()
            target_id = target[0] if len(target) >= 1 else None
            target_start = int(target[1]) if len(target) >= 2 else None
            target_end = int(target[2]) if len(target) >= 3 else None
            qlen = query_len.get(target_id)

            query_cov = None
            if qlen and target_start and target_end:
                query_cov = (abs(target_end - target_start) + 1) / qlen

            mrnas.append({
                "window": seqid,
                "mrna_id": attrs.get("ID"),
                "start": int(start),
                "end": int(end),
                "strand": strand,
                "score": float(score) if score != "." else None,
                "rank": int(attrs.get("Rank", 999999)),
                "identity": float(attrs.get("Identity", "nan")),
                "positive": float(attrs.get("Positive", "nan")),
                "target_protein": target_id,
                "target_start": target_start,
                "target_end": target_end,
                "query_length": qlen,
                "query_coverage": query_cov,
            })

df = pl.DataFrame(mrnas).with_columns(
    pl.col("mrna_id").replace(cds_count).cast(pl.Int64, strict=False).alias("cds_count")
)

candidates = (
    df
    .filter(
        (pl.col("query_coverage") >= 0.5) &
        (pl.col("identity") >= 0.3) &
        (pl.col("score") >= 1000)
    )
    .sort(
        ["window", "score", "query_coverage", "positive", "identity"],
        descending=[False, True, True, True, True],
    )
)

best_per_window = candidates.group_by("window", maintain_order=True).first()

candidates.write_csv(
    gff.with_name("miniprot_spidroin_mrna_candidates.tsv"),
    separator="\t",
)

best_per_window.write_csv(
    gff.with_name("miniprot_spidroin_best_mrna_per_window.tsv"),
    separator="\t",
)

best_per_window

## Step 6: pyGenomeTracks Per-Window Plots

Build structured miniprot mRNA models, convert the window miniprot output plus NHMMER/miniprot evidence to GTF tracks, and render one pyGenomeTracks figure per candidate window. The window-miniprot mRNA tracks are ordered by descending `Positive`, then protein length; each plot uses the tight interval covering all plotted records plus 1 kb on both sides. The track label area is widened and per-mRNA track height is increased so dense windows have readable right-side labels.


In [ ]:
miniprot_models_jsonl = OUT_DIR / "miniprot_window_mrna_models.jsonl"
miniprot_models_summary = OUT_DIR / "miniprot_window_mrna_summary.tsv"

parse_models_cmd = (
    "pixi run python -m spider_silkome_module.parse_miniprot_window_gff "
    f"{shell_path(miniprot_mixed)} {shell_path(window_fasta)} {shell_path(QUERY_PROTEINS)} "
    f"{shell_path(miniprot_models_jsonl)} {shell_path(miniprot_models_summary)}"
)
run_cmd(parse_models_cmd, [miniprot_models_jsonl, miniprot_models_summary], force=True)

models_summary_df = pl.read_csv(miniprot_models_summary, separator="\t", infer_schema_length=2000)
print(f"parsed miniprot mRNA models: {models_summary_df.height}")
models_summary_df.head(10)

In [ ]:
PYGENOMETRACKS_DIR = OUT_DIR / "pygenometracks"
PYGENOMETRACKS_COMMANDS = PYGENOMETRACKS_DIR / "pygenometracks_commands.tsv"
MIN_POSITIVE_FILTER = None  # Set to e.g. 0.6 to filter only the window miniprot mRNA track.
REGION_FLANK = 1_000
CLIP_REGION_TO_WINDOW = True
TRACK_LABEL_FRACTION = 0.35
PER_MRNA_TRACK_HEIGHT = 0.65
PYGENOMETRACKS_FONT_SIZE = 7
PYGENOMETRACKS_WIDTH = 42
PYGENOMETRACKS_FORCE = True

positive_arg = "" if MIN_POSITIVE_FILTER is None else f" --min-positive {MIN_POSITIVE_FILTER}"
clip_arg = " --clip-region-to-window" if CLIP_REGION_TO_WINDOW else " --no-clip-region-to-window"
build_tracks_cmd = (
    "pixi run python -m spider_silkome_module.build_pygenometracks_window_tracks "
    f"{shell_path(miniprot_models_jsonl)} {shell_path(NHMMER_GFF)} "
    f"{shell_path(MINIPROT_EVIDENCE_GFF)} {shell_path(merged_bed)} "
    f"{shell_path(OUT_DIR)}{positive_arg} --region-flank {REGION_FLANK}{clip_arg} "
    f"--track-label-fraction {TRACK_LABEL_FRACTION} "
    f"--per-mrna-track-height {PER_MRNA_TRACK_HEIGHT} "
    f"--plot-font-size {PYGENOMETRACKS_FONT_SIZE} "
    f"--plot-width {PYGENOMETRACKS_WIDTH}"
)
run_cmd(build_tracks_cmd, [PYGENOMETRACKS_COMMANDS], force=True)

pygt_commands_df = pl.read_csv(PYGENOMETRACKS_COMMANDS, separator="\t")
print(f"pyGenomeTracks windows: {pygt_commands_df.height}")
pygt_commands_df.select([
    "window_id",
    "full_region",
    "plot_region",
    "track_label_fraction",
    "font_size",
    "plot_width",
    "per_mrna_track_height",
    "output_png",
]).head(10)


In [ ]:
for row in pygt_commands_df.iter_rows(named=True):
    output_png = Path(row["output_png"])
    run_cmd(f"pixi run {row['command']}", [output_png], force=PYGENOMETRACKS_FORCE)

plot_paths = sorted((PYGENOMETRACKS_DIR / "plots").glob("*.png"))
print(f"pyGenomeTracks plots generated: {len(plot_paths)}")
for path in plot_paths[:5]:
    print(path)